# E207 Final: Fast Music Retrieval Under Time Warping

Collaborators: Jordan Carlin, Jasper Cox, Thomas Lilygren

Description


In [41]:
%matplotlib inline

In [51]:
# Configuration
N_FFT = 2048
HOP_LENGTH = 512

In [43]:
import numpy as np
import librosa as lb
import matplotlib.pyplot as plt
import IPython.display as ipd
import scipy.io as sio
import scipy.signal
import glob
import os.path
import subprocess
import pickle
from pathlib import Path
from typing import Literal
from sklearn.decomposition import non_negative_factorization as NMF

In [103]:
def download_gdrive_folder(folder_url: str, output_dir: str | Path = "./songs", quiet: bool = False) -> Path:
    """Download all files from a public Google Drive folder into output_dir.

    The Google Drive folder must have viewer access enabled for anyone with the link.
    """

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    import gdown

    gdown.download_folder(url=folder_url, output=str(output_path), quiet=quiet, use_cookies=False)
    print(f"Downloaded files to: {output_path.resolve()}")
    return output_path

In [107]:
if not Path("./songs").exists():
    download_gdrive_folder("https://drive.google.com/drive/folders/1Kl0zblH1Z6bIoyika4yuyrYwzI12SHpU")
else:
    print("Songs already downloaded. Delete songs directory to re-download new version if needed.")

Songs already downloaded. Delete songs directory to re-download new version if needed.


In [44]:
def load_audio(audio_file: Path) -> tuple[np.ndarray, int | float]:
    """Load an audio file. Returns a tuple of the audio time series and the sample rate."""
    audio, sr = lb.load(audio_file, sr=None)
    return audio, sr

In [45]:
def time_to_freq(audio: np.ndarray, sr: int | float, algorithim: Literal["stft", "cqt", "logmel", "chroma"] = "stft", n_fft: int = N_FFT, hop_length: int = HOP_LENGTH) -> np.ndarray:
    """Convert a time-domain audio signal to a frequency-domain representation."""
    match algorithim:
        case "stft":
            X = lb.stft(audio, n_fft=n_fft, hop_length=hop_length)
            return X
        case "cqt":
            C = lb.cqt(audio, sr=sr, hop_length=hop_length)
            return C
        case "logmel":
            S = lb.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length)
            log_S = lb.power_to_db(S, ref=np.max)
            return log_S
        case "chroma":
            C = lb.feature.chroma_stft(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length)
            return C
        case _:
            raise ValueError(f"Unsupported type: {algorithim}")

In [61]:
def midi_to_freq(p):
    """turn a midi number into a freq in Hz"""
    return 440 * np.power(2, (p - 69) / 12)


def init_W(N, sr, midi_notes, max_harmonics=16):
    """Create an initial W matrix with harmonic templates for each MIDI note.

    Each note gets a template with its fundamental and harmonics (up to max_harmonics)
    placed at the appropriate STFT frequency bins using Gaussian peaks for smooth
    frequency-bin assignment.

    Args:
        N: number of samples in an STFT frame (n_fft)
        sr: sample rate
        midi_notes: list of MIDI note numbers
        max_harmonics: maximum number of harmonics per note (default 16)
    Returns:
        W0: template matrix of shape (K, len(midi_notes))
    """
    K = int(N // 2 + 1)
    W0 = np.zeros((K, len(midi_notes)))
    freqs_hz = midi_to_freq(np.array(midi_notes, dtype=float))
    bin_freqs = np.arange(K) * sr / N  # frequency of each STFT bin

    for i, f0 in enumerate(freqs_hz):
        for h in range(1, max_harmonics + 1):
            fh = f0 * h
            if fh > sr / 2:
                break
            # Place a Gaussian peak centered on the harmonic frequency
            # Width proportional to f0 (narrower for low notes, wider for high)
            sigma = max(f0 * 0.03, sr / N)  # at least 1 bin wide
            peak = np.exp(-0.5 * ((bin_freqs - fh) / sigma) ** 2)
            W0[:, i] += peak / h  # weight by 1/harmonic_number

    return W0

In [62]:
MIDI_MIN = 21  # A0, lowest piano note
MIDI_MAX = 108  # C8, highest piano note
MIDI_NOTES = list(range(MIDI_MIN, MIDI_MAX + 1))

EPS = np.finfo(np.float64).eps  # small constant to avoid division by zero


def nmf(X: np.ndarray, sr: int | float, midi_notes: list[int] = MIDI_NOTES, n_fft: int = N_FFT, max_iter: int = 300) -> tuple[np.ndarray, np.ndarray]:
    """Perform NMF with fixed W (harmonic templates) using KL-divergence multiplicative updates.

    Only H (activations) is learned. W is kept as the physics-based harmonic templates.
    """
    V = np.abs(X).astype(np.float64)
    W = init_W(N=n_fft, sr=sr, midi_notes=midi_notes).astype(np.float64)

    # Normalize columns of W so no note template dominates by sheer norm
    col_norms = np.linalg.norm(W, axis=0, keepdims=True)
    col_norms[col_norms == 0] = 1.0
    W /= col_norms

    # Initialize H
    n_notes = len(midi_notes)
    H = np.random.rand(n_notes, V.shape[1]).astype(np.float64) + EPS

    # Precompute W column sums (used in KL update denominator)
    W_col_sum = W.sum(axis=0, keepdims=True).T  # (n_notes, 1)

    for _ in range(max_iter):
        WH = W @ H + EPS
        H *= (W.T @ (V / WH)) / (W_col_sum + EPS)

    return W, H

In [56]:
def extract_highest_energy(H: np.ndarray, threshold: float = 0.1, median_frames: int = 7) -> np.ndarray:
    """For each time frame, return the index of the highest-activation component.

    Applies median filtering across time to smooth out frame-level noise.
    Frames where the peak activation is below `threshold` * global max are masked as -1 (silence).
    """
    # Smooth activations along time axis to reduce spurious jumps
    H_smooth = scipy.signal.medfilt2d(H.astype(np.float64), kernel_size=(1, median_frames))
    dominant = np.argmax(H_smooth, axis=0)
    peak_per_frame = H_smooth[dominant, np.arange(H_smooth.shape[1])]
    silence_mask = peak_per_frame < threshold * peak_per_frame.max()
    dominant[silence_mask] = -1
    return dominant

In [57]:
def test_single_file(audio_file: Path) -> None:
    audio, sr = load_audio(audio_file)
    X = time_to_freq(audio, sr, "stft")
    W, H = nmf(X, sr)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im0 = axes[0].imshow(W, aspect="auto", origin="lower", interpolation="none")
    axes[0].set_title("W (Basis Vectors / MIDI Templates)")
    axes[0].set_xlabel("MIDI Note (21=A0 ... 108=C8)")
    axes[0].set_ylabel("Frequency Bin")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(H, aspect="auto", origin="lower", interpolation="none")
    axes[1].set_title("H (Piano Roll)")
    axes[1].set_xlabel("Time Frame")
    axes[1].set_ylabel("MIDI Note Index")
    fig.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    plt.show()

    dominant_component = extract_highest_energy(H)
    active = dominant_component >= 0
    dominant_midi = np.array(MIDI_NOTES)[dominant_component[active]]
    times = lb.frames_to_time(np.where(active)[0], sr=sr, hop_length=HOP_LENGTH)
    plt.figure(figsize=(12, 4))
    plt.scatter(times, dominant_midi, s=5)
    plt.xlabel("Time (s)")
    plt.ylabel("MIDI Note")
    plt.title("Dominant Note Over Time")
    plt.yticks(range(MIDI_MIN, MIDI_MAX + 1, 12), [lb.midi_to_note(m) for m in range(MIDI_MIN, MIDI_MAX + 1, 12)])
    plt.show()

In [ ]:
test_single_file(Path("./references/beethoven.mp3"))